# 04 · Ajuste de hiperparámetros

Usa los candidatos registrados por 03. El espacio de regresión logística abarca C de 0,001 a 100. La tabla muestra el rango efectivo, además de las mejores configuraciones.

In [ ]:
EJECUTAR_TUNING = False

In [2]:
import json
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")
cliente = MlflowClient()
NOMBRE_EXPERIMENTO = "Sistema de alerta temprana de volatilidad para acciones"
NOMBRE_REGISTRO = "alerta-volatilidad-aapl-modelo"
ruta_revision = raiz / "docs" / "revision_runs.json"

def guardar_ids(contenido):
    temporal = ruta_revision.with_suffix(".tmp")
    ruta_revision.parent.mkdir(exist_ok=True)
    temporal.write_text(json.dumps(contenido, indent=2), encoding="utf-8")
    temporal.replace(ruta_revision)

revision = (
    json.loads(ruta_revision.read_text(encoding="utf-8"))
    if ruta_revision.is_file() else {}
)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '673213bc5b0fe2ef795cd2492ece28723d4b3d92656aadfb5cadb71ed13dbafb', 'filas_snapshot': 4087}


In [4]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

Desarrollo: (3212, 18)
Fechas reservadas para prueba: 805


In [5]:
def evaluar_candidato(estimador):
    resultados = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        X_train = X_desarrollo.iloc[idx_train]
        X_val = X_desarrollo.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(0.80)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(clone(estimador), columnas_numericas)
        inicio = perf_counter()
        pipeline.fit(X_train, y_train)
        segundos = perf_counter() - inicio

        auc_train = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
        auc_val = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

        resultados.append({
            "pliegue": pliegue,
            "fin_train": X_train.index[-1].date(),
            "inicio_validacion": X_val.index[0].date(),
            "fin_validacion": X_val.index[-1].date(),
            "p80_train": p80,
            "roc_auc_train": auc_train,
            "roc_auc_val": auc_val,
            "brecha_roc_auc": auc_train - auc_val,
            "segundos_ajuste": segundos,
        })

    return pd.DataFrame(resultados)

In [6]:
from scipy.stats import loguniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from sklearn.svm import SVC

estimadores = {
    "RandomForestClassifier": RandomForestClassifier(random_state=SEMILLA),
    "LogisticRegression": LogisticRegression(random_state=SEMILLA),
    "SVC": SVC(probability=True, random_state=SEMILLA),
}

espacios = {
    "RandomForestClassifier": {
        "n_estimators": [80, 120, 160],
        "max_depth": [3, 5, 8, 12, None],
        "min_samples_leaf": [2, 5, 10, 20],
        "max_features": ["sqrt", 0.5],
    },
    "LogisticRegression": {
        "C": loguniform(1e-3, 1e2),
    },
    "SVC": {
        "C": loguniform(1e-2, 1e2),
        "gamma": loguniform(1e-4, 1),
    },
}

familias = {
    "RandomForestClassifier": "bagging",
    "LogisticRegression": "lineal_regularizado",
    "SVC": "maquina_vectores_soporte",
}

In [7]:
import subprocess

import mlflow.sklearn
from mlflow.models import infer_signature


def ajustar_modelo(referencia):
    nombre = referencia["modelo"]
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    configuraciones = list(ParameterSampler(
        espacios[nombre], n_iter=30, random_state=SEMILLA
    ))

    tags = {
        "fase": "tuning",
        "familia": familias[nombre],
        "git_commit": commit,
    }
    datos_tuning = {
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "run_id_default": referencia["run_id"],
        "semilla_busqueda": SEMILLA,
        "n_pliegues": cv_temporal.n_splits,
        "gap": cv_temporal.gap,
        "cuantil_target": 0.80,
    }
    filas = []

    with mlflow.start_run(
        experiment_id=experimento.experiment_id,
        run_name=f"tuning_{nombre}",
    ) as padre:
        mlflow.set_tags(tags)
        mlflow.log_params({**datos_tuning, "n_configuraciones": len(configuraciones)})

        for numero, parametros in enumerate(configuraciones, start=1):
            candidato = clone(estimadores[nombre]).set_params(**parametros)

            with mlflow.start_run(
                run_name=f"{nombre}_{numero:02d}", nested=True
            ) as hija:
                mlflow.set_tags(tags)
                mlflow.log_params({
                    **datos_tuning,
                    "configuracion": numero,
                    "semilla_modelo": SEMILLA,
                    **{f"modelo__{k}": v for k, v in candidato.get_params().items()},
                })

                resultados = evaluar_candidato(candidato)
                metricas = {
                    "roc_auc_train_media_cv": float(resultados["roc_auc_train"].mean()),
                    "roc_auc_val_media_cv": float(resultados["roc_auc_val"].mean()),
                    "roc_auc_val_std_cv": float(resultados["roc_auc_val"].std(ddof=1)),
                    "brecha_roc_auc_cv": float(resultados["brecha_roc_auc"].mean()),
                    "segundos_ajuste_cv_total": float(resultados["segundos_ajuste"].sum()),
                }

                mlflow.log_metrics(metricas)
                mlflow.log_text(
                    resultados.to_csv(index=False), "resultados_por_pliegue.csv"
                )
                filas.append({
                    "configuracion": numero,
                    "run_id": hija.info.run_id,
                    **parametros,
                    **metricas,
                })

            print(
                f"{nombre} {numero}/{len(configuraciones)} · "
                f"AUC {metricas['roc_auc_val_media_cv']:.4f}",
                flush=True,
            )

        resumen = pd.DataFrame(filas).sort_values(
            "roc_auc_val_media_cv", ascending=False, kind="stable"
        )
        mejor = resumen.iloc[0]
        parametros_mejor = configuraciones[int(mejor["configuracion"]) - 1]

        auc_default = float(referencia["auc_default"])
        auc_ajustado = float(mejor["roc_auc_val_media_cv"])
        std_ajustado = float(mejor["roc_auc_val_std_cv"])
        mejora = auc_ajustado - auc_default
        mejora_segun_pset = mejora > 0 and mejora >= std_ajustado

        mlflow.log_metrics({
            "roc_auc_default": auc_default,
            "roc_auc_default_std_cv": float(referencia["std_default"]),
            "roc_auc_val_media_cv": auc_ajustado,
            "roc_auc_val_std_cv": std_ajustado,
            "mejora_auc": mejora,
        })
        mlflow.set_tags({
            "mejor_run_hijo": mejor["run_id"],
            "mejora_segun_pset": str(mejora_segun_pset).lower(),
        })
        mlflow.log_dict(parametros_mejor, "mejores_parametros.json")
        mlflow.log_text(resumen.to_csv(index=False), "resumen_tuning.csv")

        p80_desarrollo = float(vol_desarrollo.quantile(0.80))
        y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)
        mlflow.log_param("p80_desarrollo", p80_desarrollo)

        mejor_estimador = clone(estimadores[nombre]).set_params(**parametros_mejor)
        pipeline = crear_pipeline(mejor_estimador, columnas_numericas)

        inicio = perf_counter()
        pipeline.fit(X_desarrollo, y_desarrollo)
        mlflow.log_metric("segundos_reajuste_desarrollo", perf_counter() - inicio)

        ejemplo = X_desarrollo.head(5).copy()
        modelo_registrado = mlflow.sklearn.log_model(
            pipeline,
            name=f"pipeline_{nombre}_ajustado",
            signature=infer_signature(ejemplo, pipeline.predict_proba(ejemplo)),
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print(f"Run padre: {padre.info.run_id}")
    print(f"URI del modelo: {modelo_registrado.model_uri}")
    print(f"Default: {auc_default:.4f} ± {referencia['std_default']:.4f}")
    print(f"Ajustado: {auc_ajustado:.4f} ± {std_ajustado:.4f}")
    print(f"Mejora: {mejora:+.4f} | Cumple criterio PSet: {mejora_segun_pset}")
    display(resumen.head())
    return resumen, padre.info.run_id

In [8]:
if EJECUTAR_TUNING:
    assert revision.get("sha256_dataset_desarrollo") == hash_desarrollo, (
        "Registra primero los candidatos de 03 con esta extracción."
    )
    # Selección de tres modelos

    import mlflow

    experimento = cliente.get_experiment_by_name(NOMBRE_EXPERIMENTO)
    ids_candidatos = revision["candidatos"]
    cliente = mlflow.MlflowClient()
    filas = []

    for run_id in ids_candidatos:
        run = cliente.get_run(run_id)
        if (
            run.info.status != "FINISHED"
            or run.info.experiment_id != experimento.experiment_id
            or run.data.tags.get("fase") != "candidato"
        ):
            raise ValueError(f"Revisa la corrida candidata {run_id}")

        filas.append({
            "run_id": run_id,
            "modelo": run.data.params["modelo"],
            "auc_default": run.data.metrics["roc_auc_val_media_cv"],
            "std_default": run.data.metrics["roc_auc_val_std_cv"],
        })

    ranking = pd.DataFrame(filas).sort_values("auc_default", ascending=False)
    assert ranking["modelo"].nunique() == 5, "Selecciona una corrida por candidato"

    seleccionados = ranking.head(3)
    display(ranking)
    ids_tuning_revision = {}
    for _, referencia in seleccionados.iterrows():
        corrida = cliente.get_run(referencia["run_id"])
        assert corrida.data.params["sha256_dataset_desarrollo"] == hash_desarrollo

    for _, referencia in seleccionados.iterrows():
        resumen, id_padre = ajustar_modelo(referencia)
        ids_tuning_revision[referencia["modelo"]] = id_padre
        if referencia["modelo"] == "LogisticRegression":
            print("Rango efectivo C:", resumen["C"].min(), resumen["C"].max())

    guardar_ids({**revision, "tuning": ids_tuning_revision})
    print("IDs para 05:", ids_tuning_revision)
else:
    print('Tuning desactivado. Habilítalo cuando 03 haya registrado los candidatos.')

,run_id,modelo,auc_default,std_default
1,432e8271a62e42389de9d59e8d934d48,RandomForestClassifier,0.722572,0.047955
0,f7e8fd5b27da4cc890134287aae13786,LogisticRegression,0.709873,0.067944
4,b6350a1f8f5945668664408312e7d68f,SVC,0.699474,0.040146
2,d6f267f4c60346c58462eaa0af75ce95,HistGradientBoostingClassifier,0.651441,0.049464
3,95f5bca094da491aacf43ce1da4349ce,MLPClassifier,0.636092,0.075016


🏃 View run RandomForestClassifier_01 at: http://127.0.0.1:5000/#/experiments/0/runs/c50f3fed1a9642e5a34fd52db0bf3489
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
RandomForestClassifier 1/30 · AUC 0.7295
🏃 View run RandomForestClassifier_02 at: http://127.0.0.1:5000/#/experiments/0/runs/6ab27dadb64f44a68a55323bbdd4a3cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
RandomForestClassifier 2/30 · AUC 0.7332
🏃 View run RandomForestClassifier_03 at: http://127.0.0.1:5000/#/experiments/0/runs/d30607b48cd94a129389ca66a004b997
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
RandomForestClassifier 3/30 · AUC 0.7461
🏃 View run RandomForestClassifier_04 at: http://127.0.0.1:5000/#/experiments/0/runs/052f2ec77cb04685b3274c87b855b7d2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
RandomForestClassifier 4/30 · AUC 0.7370
🏃 View run RandomForestClassifier_05 at: http://127.0.0.1:5000/#/experiments/0/runs/62e8957de0cb4dc8ba949a2449a98da9
🧪 View expe

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 19:06:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run tuning_RandomForestClassifier at: http://127.0.0.1:5000/#/experiments/1/runs/28f9e653515d49b58784d3c29e5a9b2f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run padre: 28f9e653515d49b58784d3c29e5a9b2f
URI del modelo: models:/m-a42ca0e4f9204435a231ac0aadf01e2c
Default: 0.7226 ± 0.0480
Ajustado: 0.7478 ± 0.0625
Mejora: +0.0252 | Cumple criterio PSet: False


,configuracion,run_id,n_estimators,min_samples_leaf,max_features,max_depth,roc_auc_train_media_cv,roc_auc_val_media_cv,roc_auc_val_std_cv,brecha_roc_auc_cv,segundos_ajuste_cv_total
12,13,04aed57cec544d3e8d703335b499df27,160,20,sqrt,3.0,0.793892,0.747782,0.062487,0.046111,4.735407
7,8,49357e7c050a48c5a6c7d4a9e7becc84,120,20,sqrt,3.0,0.795209,0.747650,0.061575,0.047559,3.135739
2,3,d30607b48cd94a129389ca66a004b997,120,5,sqrt,3.0,0.806823,0.746080,0.065830,0.060743,3.310955
17,18,5971a788135a4476be3d199400cdc204,80,2,sqrt,3.0,0.804797,0.745921,0.062863,0.058875,2.631905
22,23,1fc9f633c6d24687b757fe83006a0e15,120,10,sqrt,5.0,0.858713,0.742733,0.064544,0.115981,4.446817


🏃 View run LogisticRegression_01 at: http://127.0.0.1:5000/#/experiments/0/runs/c02f36e92c9a40dfa605c9bc89f1a58a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
LogisticRegression 1/30 · AUC 0.7245
🏃 View run LogisticRegression_02 at: http://127.0.0.1:5000/#/experiments/0/runs/675b357a35bc442da031981ca88df2fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
LogisticRegression 2/30 · AUC 0.7045
🏃 View run LogisticRegression_03 at: http://127.0.0.1:5000/#/experiments/0/runs/d1850e0b7ff24054992b928925432376
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
LogisticRegression 3/30 · AUC 0.7059
🏃 View run LogisticRegression_04 at: http://127.0.0.1:5000/#/experiments/0/runs/2ce45e3fb5bc4fb2b04ef09ab2a1277e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
LogisticRegression 4/30 · AUC 0.7099
🏃 View run LogisticRegression_05 at: http://127.0.0.1:5000/#/experiments/0/runs/089fd9dfe6064bd9976519753896ee3a
🧪 View experiment at: http://127.0.0.1:5000/#/e

... (4 outputs left out)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/27 19:19:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run tuning_SVC at: http://127.0.0.1:5000/#/experiments/1/runs/7b6aac14ada74df7b79b034cc4debba6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run padre: 7b6aac14ada74df7b79b034cc4debba6
URI del modelo: models:/m-8668f59751b84ec9a1124d32b31f8251
Default: 0.6995 ± 0.0401
Ajustado: 0.7298 ± 0.0691
Mejora: +0.0303 | Cumple criterio PSet: False


,configuracion,run_id,C,gamma,roc_auc_train_media_cv,roc_auc_val_media_cv,roc_auc_val_std_cv,brecha_roc_auc_cv,segundos_ajuste_cv_total
7,8,41b2704656ee4813bb69c752fc331fb1,0.053370,0.000542,0.748199,0.729794,0.069125,0.018405,4.111076
10,11,8b0f30a9db6246eda8c11e436ad100aa,2.801635,0.000361,0.775278,0.729219,0.054644,0.046059,4.419397
24,25,4d535a52a55a4c68a5a34a7240f8b9cc,1.537592,0.000549,0.773751,0.724395,0.054269,0.049356,4.690509
9,10,0d0dbf8f751c442b9779d8f73440ede2,0.534294,0.001462,0.777453,0.723860,0.060367,0.053592,4.496491
29,30,456e5ca72b274cc9bd3e4386529227c7,0.015167,0.002001,0.756476,0.723857,0.070517,0.032619,3.895618


IDs para 05: {'RandomForestClassifier': '28f9e653515d49b58784d3c29e5a9b2f', 'LogisticRegression': 'b5f0bb696a9a456192ee79755b853c53', 'SVC': '7b6aac14ada74df7b79b034cc4debba6'}
